# 🇮🇳 Fine-Tuning Mistral-7B-Instruct on Hinglish Conversations using QLoRA (PEFT)

Welcome to this end-to-end tutorial on fine-tuning **`mistralai/Mistral-7B-Instruct-v0.3`** for natural, culturally attuned **Romanized Hinglish** (Hindi code-mixed with English in Latin script) using **4-bit Parameter-Efficient Fine-Tuning (QLoRA)**.

---

### 🎯 Objectives & Hardware Optimization:
1. **High Fine-Tuning Delta**: Mistral-7B has a strong English/code foundation but struggles with colloquial Romanized Hinglish out-of-the-box, making the post-training adaptation remarkably stark and effective.
2. **Kaggle P100 GPU Optimization**: Configured for 16GB VRAM Pascal GPU using **4-bit NF4 Quantization (QLoRA)** and **FP16 compute** (`use_fp16: true`), fitting the 7B model comfortably in **~5.5 GB VRAM** with zero OOM risk.
3. **Clean Architecture**: Built on modular components (`ModelBuilder`, `DataLoader`, `Trainer`, `InferenceEngine`).
4. **Before vs. After Comparison**: Directly compare responses from the **Un-finetuned Base Mistral-7B** vs. the **Fine-Tuned QLoRA Model** on identical benchmark prompts.

## 1. Environment Setup & Hardware Verification

Install dependencies and verify GPU availability (NVIDIA P100 / T4).

In [ ]:
# Install required dependencies (uncomment when running in Kaggle or Colab)
# !pip install -q "transformers>=4.46.0" "peft>=0.13.0" "trl<0.15.0" "accelerate>=0.34.0" "bitsandbytes>=0.43.0" datasets pyarrow pyyaml huggingface_hub matplotlib

import os
import sys
import torch
import yaml
from huggingface_hub import login

# Add project root to sys.path
current_dir = os.path.abspath('')
project_root = os.path.dirname(current_dir) if os.path.basename(current_dir) == 'lora' else current_dir
if project_root not in sys.path:
    sys.path.insert(0, project_root)

print(f'Project root: {project_root}')
print(f'PyTorch Version: {torch.__version__}')
print(f'CUDA Available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    device_count = torch.cuda.device_count()
    print(f'GPU Count: {device_count}')
    for i in range(device_count):
        props = torch.cuda.get_device_properties(i)
        print(f'  [GPU {i}]: {props.name} | VRAM: {props.total_memory / 1e9:.2f} GB')

### 🔑 Hugging Face Authentication (Kaggle & Colab Compatible)

Authenticate with Hugging Face Hub to access datasets and upload fine-tuned adapters.

In [ ]:
hf_token = os.environ.get('HF_TOKEN')

# 1. Try Kaggle Secrets
if not hf_token:
    try:
        from kaggle_secrets import UserSecretsClient
        user_secrets = UserSecretsClient()
        hf_token = user_secrets.get_secret('HF_TOKEN')
    except Exception:
        pass

# 2. Try Google Colab Secrets
if not hf_token:
    try:
        from google.colab import userdata
        hf_token = userdata.get('HF_TOKEN')
    except Exception:
        pass

# 3. Fallback or Login
if hf_token:
    os.environ['HF_TOKEN'] = hf_token
    login(token=hf_token)
    print('✅ Authenticated with Hugging Face successfully!')
else:
    print('ℹ️ No HF_TOKEN detected. Continuing in public/read-only mode.')

## 2. Load Configuration & Hardware Settings

We load project settings from `lora/config.yaml`.

In [ ]:
config_path = os.path.join(project_root, 'lora', 'config.yaml')
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Ensure Kaggle P100 FP16 precision
native_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported(including_emulation=False)
config['use_bf16'] = False if not native_bf16 else config.get('use_bf16', False)
config['use_fp16'] = True if not native_bf16 else config.get('use_fp16', True)

print(f'⚡ Target Model: {config["model_id"]}')
print(f'⚡ 4-Bit QLoRA: {config.get("use_4bit", True)}')
print(f'⚡ Precision: FP16={config["use_fp16"]}, BF16={config["use_bf16"]}')

print('\n--- Active Training Configuration ---')
for k, v in config.items():
    print(f'{k}: {v}')

## 3. 🔍 Baseline Evaluation: Base Mistral-7B (Before Fine-Tuning)

Let us evaluate the **out-of-the-box base model (`mistralai/Mistral-7B-Instruct-v0.3`)** on 4 representative Hinglish prompts covering daily life, cooking, career/tech decisions, and interpersonal advice.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

test_prompts = [
    'arre yaar, aaj office mein itna zyada kaam tha ki dimag ka dahi ho gaya. thoda relax karne ka koi mast idea batao.',
    'weekend pe doston ke saath kuch accha cook karna hai. koi easy aur spicy street food style recipe suggest karo na!',
    'bhai ek confusion hai, kya mujhe apna personal laptop upgrade karna chahiye ya cloud setup use karna chahiye data science ke liye?',
    'yaar mera dost bina bataye mujhse naraz ho gaya hai, samajh nahi aa raha kaise baat shuru karun.'
]

print('Loading Base Mistral-7B in 4-bit for baseline evaluation...')
torch_dtype = torch.float16
base_tokenizer = AutoTokenizer.from_pretrained(config['model_id'])
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch_dtype,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    config['model_id'],
    quantization_config=quant_config,
    device_map='auto',
    trust_remote_code=True
)
base_model.eval()

base_responses = []
system_prompt = config.get('system_prompt', 'You are a helpful assistant.')

print('\n================ BASE MODEL RESPONSES (BEFORE FINE-TUNING) ================')
for i, prompt in enumerate(test_prompts, 1):
    messages = [
        {'role': 'system', 'content': system_prompt},
        {'role': 'user', 'content': prompt}
    ]
    prompt_str = base_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = base_tokenizer(prompt_str, return_tensors='pt').to(base_model.device)
    
    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=256,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=base_tokenizer.eos_token_id
        )
    
    gen_text = base_tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True).strip()
    base_responses.append(gen_text)
    
    print(f'\n[Prompt {i}]: {prompt}')
    print(f'[Base Model Output]:\n{gen_text}')
    print('-' * 70)

# Unload base model to free up GPU memory for training
del base_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('\n🧹 Base model unloaded from GPU memory.')

## 4. Dataset Loading & Chat Formatting

We load the curated dataset [`theguywithblacktie/hinglish-conversations`](https://huggingface.co/datasets/theguywithblacktie/hinglish-conversations) using `DataLoader`.

In [ ]:
from lora.src.data_loader import DataLoader

print('Loading and formatting Hinglish dataset...')
tokenizer = AutoTokenizer.from_pretrained(config['model_id'])
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

data_loader = DataLoader(config, tokenizer)
train_data, val_data = data_loader.load_and_prepare()

print(f'\n✅ Train Dataset Size: {len(train_data):,} samples')
print(f'✅ Validation Dataset Size: {len(val_data):,} samples')

print('\n--- Sample Formatted Training Turn ---')
print(train_data[0]['text'][:500] + '...')

## 5. Model Architecture & QLoRA Setup

We use `ModelBuilder` with 4-bit BitsAndBytes quantization to freeze Mistral-7B and inject trainable low-rank adapters ($r=16, \alpha=32$).

In [ ]:
from lora.src.model_builder import ModelBuilder

print('Building QLoRA model and injecting adapters...')
model_builder = ModelBuilder(config)
model, tokenizer = model_builder.build()
model.enable_input_require_grads()

## 6. Supervised Fine-Tuning Loop

Execute the training loop using `Trainer`. Upon completion, it automatically saves adapters to `./mistral-lora-outputs/`.

In [ ]:
from lora.src.trainer import Trainer

trainer = Trainer(
    config=config,
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_data,
    val_dataset=val_data
)

print('🚀 Starting fine-tuning loop on Kaggle P100 GPU...')
trainer.train_and_save()
print('🎉 Training complete! Artifacts saved to:', config.get('output_dir', './mistral-lora-outputs'))

## 7. 🚀 Post-Training Evaluation: Testing Fine-Tuned QLoRA Model

We load our newly fine-tuned LoRA adapters using `InferenceEngine` and run inference on the exact same 4 benchmark test prompts.

In [ ]:
from lora.src.inference import InferenceEngine

# Free training objects from GPU memory
del model, trainer
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print('Loading Fine-Tuned Mistral-7B QLoRA Model for inference...')
inference_engine = InferenceEngine(config)

lora_responses = []
print('\n================ FINE-TUNED MODEL RESPONSES (AFTER QLoRA) ================')
for i, prompt in enumerate(test_prompts, 1):
    resp = inference_engine.generate(
        user_prompt=prompt,
        system_prompt=config.get('system_prompt', 'You are a helpful assistant.')
    )
    lora_responses.append(resp)
    print(f'\n[Prompt {i}]: {prompt}')
    print(f'[QLoRA Model Output]:\n{resp}')
    print('-' * 70)

## 8. 📊 Side-by-Side Comparison & Script Purity Evaluation

We evaluate character-level Latin script adherence with `ScriptPurityEvaluator` and render the side-by-side comparison cards.

In [ ]:
from IPython.display import display, HTML
from evaluation.script_purity import ScriptPurityEvaluator

purity_evaluator = ScriptPurityEvaluator()
purity_summary = purity_evaluator.evaluate_batch(lora_responses)

print(f"\n🔤 Script Purity Results (QLoRA Output):")
print(f"  - Average Latin Purity: {purity_summary['avg_latin_purity_score']}%")
print(f"  - 100% Pure Romanized Rate: {purity_summary['pure_romanized_rate']}%")
print(f"  - Devanagari Bleed Violations: {purity_summary['total_devanagari_violations']} chars")

html_output = '<h2>🔍 Before vs. After Fine-Tuning Qualitative Comparison (Mistral-7B)</h2>'

for i in range(len(test_prompts)):
    prompt = test_prompts[i]
    base_resp = base_responses[i]
    lora_resp = lora_responses[i]
    
    html_output += f'''
    <div style="border: 1px solid #ddd; border-radius: 8px; padding: 15px; margin-bottom: 20px; font-family: -apple-system, BlinkMacSystemFont, Segoe UI, Roboto, sans-serif;">
        <div style="font-size: 15px; font-weight: 600; color: #1e40af; margin-bottom: 12px;">
            💬 Prompt {i+1}: <span style="color: #111827; font-weight: 400;">{prompt}</span>
        </div>
        <div style="display: grid; grid-template-columns: 1fr 1fr; gap: 15px;">
            <div style="background: #fff1f2; border: 1px solid #fecdd3; border-radius: 6px; padding: 12px;">
                <div style="font-weight: 700; color: #be123c; margin-bottom: 6px;">❌ Base Mistral-7B (Before)</div>
                <div style="color: #374151; font-size: 14px; white-space: pre-wrap; line-height: 1.5;">{base_resp}</div>
            </div>
            <div style="background: #f0fdf4; border: 1px solid #bbf7d0; border-radius: 6px; padding: 12px;">
                <div style="font-weight: 700; color: #15803d; margin-bottom: 6px;">✅ QLoRA Fine-Tuned (After)</div>
                <div style="color: #374151; font-size: 14px; white-space: pre-wrap; line-height: 1.5;">{lora_resp}</div>
            </div>
        </div>
    </div>
    '''

display(HTML(html_output))

## 9. 💬 Interactive Chat Loop

Test your fine-tuned Hinglish assistant interactively with your own custom prompts!

In [ ]:
custom_prompt = 'bhai, kal mera interview hai aur bohot anxiety ho rahi hai. thoda guide karo na kya karun!'

print(f'User: {custom_prompt}')
ai_response = inference_engine.generate(user_prompt=custom_prompt)
print(f'\nAI: {ai_response}')